**NAME - RUCHITA PANKAJ BODKE**\
**PRN - 202401110022**\
**BRANCH - CSE-AIML**\
**BATCH - A2**\
**COURSE - GENERATIVE AI**

# Practical No-06: Retrieval-Augmented Generation (RAG) using Pre-trained LLM

In [1]:
# Cell 1 – Install required libraries

!pip install -q transformers sentence-transformers faiss-cpu pypdf accelerate

In [3]:
# Cell 2 – Import libraries

import os
import numpy as np
import torch
import faiss

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [4]:
# Cell 3 – Upload domain-specific documents

uploaded = files.upload()

documents = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        reader = PdfReader(filename)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        documents.append({
            "filename": filename,
            "text": text
        })

    elif filename.lower().endswith(".txt"):

        with open(filename, "r", encoding="utf-8") as f:
            text = f.read()

        documents.append({
            "filename": filename,
            "text": text
        })

print("Documents loaded:", len(documents))

for doc in documents:
    print(doc["filename"], "-", len(doc["text"]), "characters")

Saving fmed-13-1742345.pdf to fmed-13-1742345.pdf
Documents loaded: 1
fmed-13-1742345.pdf - 72836 characters


In [21]:
# Cell 4 – Split documents into smaller chunks

def split_text(text, chunk_size=250, overlap=50):

    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


all_chunks = []
chunk_sources = []

for doc in documents:

    chunks = split_text(doc["text"])

    for chunk in chunks:
        all_chunks.append(chunk)
        chunk_sources.append(doc["filename"])

print("Total chunks:", len(all_chunks))

Total chunks: 51


In [22]:
# Cell 5 – Generate document embeddings

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    all_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (51, 384)


In [23]:
# Cell 6 – Create FAISS vector database

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype("float32"))

print("FAISS database created successfully!")
print("Total vectors stored:", index.ntotal)

FAISS database created successfully!
Total vectors stored: 51


In [24]:
# Cell 7 – Load the pretrained LLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("LLM loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully!
Device: cpu


In [25]:
# Cell 8 – Implement document retrieval

def retrieve_documents(question, top_k=3):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        question_embedding.astype("float32"),
        min(top_k, index.ntotal)
    )

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):

        if idx != -1:

            retrieved_chunks.append({
                "text": all_chunks[idx],
                "source": chunk_sources[idx],
                "score": float(score)
            })

    return retrieved_chunks

In [26]:
# Cell 9 – Implement LLM without RAG

def generate_without_rag(question):

    prompt = f"""
    Answer the following question clearly and accurately.

    Question: {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [27]:
# Cell 10 – Implement RAG-based answer generation

def generate_with_rag(question):

    retrieved = retrieve_documents(question, top_k=3)

    context = "\n\n".join(
        item["text"] for item in retrieved
    )

    prompt = f"""
    You are a document-based question answering assistant.

    Answer the question using the information in the context.
    Give a complete and meaningful answer.
    Do not invent facts.
    If the information is missing, say:
    Information not available in the provided document.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=4,
        early_stopping=True
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved

In [28]:
# Cell 11 – Compare responses with and without RAG

def compare_responses(question):

    print("\nQUESTION:")
    print(question)

    print("\n" + "="*60)
    print("LLM RESPONSE WITHOUT RAG")
    print("="*60)

    normal_answer = generate_without_rag(question)

    print(normal_answer)

    print("\n" + "="*60)
    print("LLM RESPONSE WITH RAG")
    print("="*60)

    rag_answer, retrieved = generate_with_rag(question)

    print(rag_answer)

    print("\n" + "="*60)
    print("RETRIEVED DOCUMENT CHUNKS")
    print("="*60)

    for i, item in enumerate(retrieved):

        print(f"\nChunk {i+1}")
        print("Source:", item["source"])
        print("Similarity:", round(item["score"], 4))
        print("Content:", item["text"][:300])

    return normal_answer, rag_answer

In [29]:
questions = [
    "What problem related to diabetic foot ulcer datasets is discussed?",
    "Why is there a shortage of large labeled DFU image datasets?",
    "How are LSTM layers used in the proposed framework?",
    "What are the limitations of the proposed framework?",
    "What future improvements are discussed in the paper?"
]

for question in questions:

    compare_responses(question)


QUESTION:
What problem related to diabetic foot ulcer datasets is discussed?

LLM RESPONSE WITHOUT RAG
amputation of the limb

LLM RESPONSE WITH RAG
The only data set available is from Lancashire Teaching Hospital

RETRIEVED DOCUMENT CHUNKS

Chunk 1
Source: fmed-13-1742345.pdf
Similarity: 0.6863
Content: are not enough large, well-labeled medical datasets. This is especially true for diabetic foot ulcer (DFU) images, as strict privacy laws and legal barriers make sharing difficult. In the case of diabetic foot ulcer (DFU) imaging, dataset scarcity is primarily attributed to several factors, which ar

Chunk 2
Source: fmed-13-1742345.pdf
Similarity: 0.6447
Content: Frontiers in Medicine 01 frontiersin.org TYPE Original Research PUBLISHED 09 March 2026 DOI 10.3389/fmed.2026.1742345 OPEN ACCESS EDITED BY Shameer Khader, Sanofi, France REVIEWED BY Birmohan Singh, Sant Longowal Institute of Engineering and Technology, India Sarad Pawar Naik Bukke, Kampala Internat

Chunk 3
Source: fmed-13-1

In [30]:
# Cell 13 – Evaluate and compare results

import pandas as pd

questions = [
    "What is the main objective of the document?",
    "What are the important methods discussed?",
    "What are the major findings?"
]

results = []

for question in questions:

    normal_answer = generate_without_rag(question)

    rag_answer, retrieved = generate_with_rag(question)

    results.append({
        "Question": question,
        "LLM Without RAG": normal_answer,
        "LLM With RAG": rag_answer,
        "Retrieved Chunks": len(retrieved),
        "Retrieved Sources": ", ".join(
            sorted(set(item["source"] for item in retrieved))
        )
    })

comparison_df = pd.DataFrame(results)

pd.set_option("display.max_colwidth", 200)

display(comparison_df)

,Question,LLM Without RAG,LLM With RAG,Retrieved Chunks,Retrieved Sources
0,What is the main objective of the document?,to be readable,Diffusion-based approaches in medical image generation and analysis.,3,fmed-13-1742345.pdf
1,What are the important methods discussed?,a) a) a) b) c) d) e) f) g) h) i) j) k) l) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m) n) m,SS,3,fmed-13-1742345.pdf
2,What are the major findings?,"The systolic velocity of the systolic current is a constant, and the systolic velocity of the systolic current is a constant.",SS,3,fmed-13-1742345.pdf


In [31]:
retrieval_results = retrieve_documents(
    "What is the main objective of this research paper?"
)

retrieval_df = pd.DataFrame([
    {
        "Document": item["source"],
        "Similarity Score": round(item["score"], 4),
        "Retrieved Content": item["text"][:200]
    }
    for item in retrieval_results
])

display(retrieval_df)

,Document,Similarity Score,Retrieved Content
0,fmed-13-1742345.pdf,0.3199,"Project administra- tion, Supervision, Writing – review & editing. SI: Writing – review & editing, Project administration, Supervision. AW: Writing – review & editing, Supervision, Project adminis..."
1,fmed-13-1742345.pdf,0.1402,"A review paper about deep learning for medical image analysis. Comput Math Methods Med . (2023) 2023:1–10. doi: 10.1155/2023/7091301 7. Hu C, Xia Y, Zheng Z, Cao M. AI-based large-scale screening ..."
2,fmed-13-1742345.pdf,0.1394,"Frank, et al Feature extraction for generative medical imaging evaluation: new evidence against an evolving trend,"" arXiv [Preprint], (2024). doi: 10.48550/arXiv.2311.13717 14. al Nomaan Nafi A., ..."
